In [1]:
import MDAnalysis as mda
import matplotlib.pyplot as plt
import numpy as np
import os
import re
from glob import glob 
import random
from braceexpand import braceexpand

ModuleNotFoundError: No module named 'braceexpand'

In [6]:
files= [("/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.xtc"),
    ("/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.xtc"),
        ("/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.xtc")]

In [3]:
def build_com_matrix(files, sel="resname BDQ", ref="resname POPE"):
    all_com = []

    for top, traj in files:
        u = mda.Universe(top, traj)
        com = get_com(u, sel, ref)
        all_com.append(com)

    # pad naar max lengte (robust!)
    max_len = max(c.shape[0] for c in all_com)

    arr = np.full((len(all_com), max_len, 3), np.nan)

    for i, com in enumerate(all_com):
        arr[i, :len(com), :] = com

    return arr

In [4]:
def get_com(u, sel: str, ref: str):
    n_frames = len(u.trajectory)
    com_arr = np.empty((n_frames, 3))

    sel_group = u.select_atoms(sel)
    ref_group = u.select_atoms(ref)

    for i, ts in enumerate(u.trajectory):
        com_sel = sel_group.center_of_mass(unwrap=True)
        com_ref = ref_group.center_of_mass()
        com_arr[i] = com_sel - com_ref

    return com_arr

In [7]:
BDN_com_all_reps = np.empty((len(files), 5001, 3))

for i, simulation in enumerate(files):
    topology, trajectory = simulation
    u = mda.Universe(topology, trajectory)
    print(u.trajectory)
    BDN_com_all_reps[i, :, :] = get_com(u, sel = "resname BDQ", ref = "resname POPE")

<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.xtc with 5001 frames of 43975 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.xtc with 5001 frames of 43903 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.xtc with 5001 frames of 43873 atoms>


In [8]:
def get_close_idx_com(com_all, target_z, tol):
    z = com_all[:, :, 2]

    idx = np.argwhere(np.isclose(z, target_z, atol=tol))
    n = len(idx)

    if n == 0:
        return None, None, 0

    choice = idx[np.random.randint(n)]
    value = z[choice[0], choice[1]]

    return value, choice, n

In [9]:
def write_frame(topology, trajectory, frame_idx, output_file):
    u = mda.Universe(topology, trajectory)
    u.trajectory[frame_idx]

    with mda.Writer(output_file, u.atoms.n_atoms) as w:
        w.write(u.atoms)

    return output_file

In [10]:
def get_atom_indices(structure_file, selection):
    u = mda.Universe(structure_file)
    sel = u.select_atoms(selection)

    return (sel.indices + 1).tolist()

In [11]:
def to_plumed_ranges(atom_list):
    ranges = []
    start = prev = atom_list[0]

    for a in atom_list[1:]:
        if a == prev + 1:
            prev = a
        else:
            ranges.append(f"{start}-{prev}" if start != prev else f"{start}")
            start = prev = a

    ranges.append(f"{start}-{prev}" if start != prev else f"{start}")
    return ",".join(ranges)

In [12]:
def write_plumed(plumedfile, where, structure_file):
    name = os.path.basename(plumedfile).replace(".dat", "")

    bdq_atoms = get_atom_indices(structure_file, "resname BDQ")
    all_atoms = get_atom_indices(structure_file, "resname POPE")

    bdq_sel = to_plumed_ranges(bdq_atoms)
    all_sel = to_plumed_ranges(all_atoms)

    with open(plumedfile, "w") as f:
        f.write("UNITS LENGTH=A\n")
        f.write(f"WHOLEMOLECULES ENTITY0={bdq_sel}\n")
        f.write(f"MOLINFO STRUCTURE={name}.pdb\n")

        f.write(f"pop: COM ATOMS={all_sel}\n")
        f.write(f"bdq: COM ATOMS={bdq_sel}\n")

        f.write("d1: DISTANCE ATOMS=pop,bdq COMPONENTS\n")

        f.write(f"RESTRAINT ARG=d1.z KAPPA=15.0 AT={where}\n")
        f.write(f"PRINT ARG=d1.z FILE={name}-colvar.dat STRIDE=1\n")

In [24]:
target_z_values = np.arange(-16, 16, 1)
tolerance = 0.1

mapping_file = "umbrella_snapshot_mapping.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+1}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+1}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+1},{z_target},{rep + 2},{frame},{value},topol{rep + 2}.top,index{rep + 2}.ndx\n")



z=-16 | rep=2 frame=886 matches=149
z=-15 | rep=6 frame=2315 matches=234
z=-14 | rep=0 frame=689 matches=357
z=-13 | rep=2 frame=2513 matches=476
z=-12 | rep=0 frame=1114 matches=451
z=-11 | rep=4 frame=4161 matches=403
z=-10 | rep=6 frame=3732 matches=326
z=-9 | rep=5 frame=4184 matches=250
z=-8 | rep=4 frame=1506 matches=147
z=-7 | rep=2 frame=1909 matches=92
z=-6 | rep=2 frame=3408 matches=59
z=-5 | rep=2 frame=2105 matches=33
z=-4 | rep=6 frame=4394 matches=18
z=-3 | rep=2 frame=4101 matches=18
z=-2 | rep=5 frame=2798 matches=12
z=-1 | rep=6 frame=4841 matches=15
z=0 | rep=0 frame=1719 matches=17
z=1 | rep=2 frame=4983 matches=16
z=2 | rep=0 frame=1745 matches=7
z=3 | rep=6 frame=4915 matches=18
z=4 | rep=0 frame=1837 matches=22
z=5 | rep=1 frame=4162 matches=43
z=6 | rep=1 frame=4587 matches=63
z=7 | rep=0 frame=1814 matches=97
z=8 | rep=5 frame=2230 matches=150
z=9 | rep=1 frame=2950 matches=188
z=10 | rep=1 frame=4004 matches=220
z=11 | rep=0 frame=4673 matches=314
z=12 | rep=5 

In [13]:
print(BDN_com_all_reps[:, :, 2].min(), BDN_com_all_reps[:, :, 2].max())

-34.055168834132694 35.6211019866044


In [19]:
target_z_values = np.arange(-32, -16, 1)
tolerance = 0.125

mapping_file = "umbrella_snapshot_mapping_bdqunbiased.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/umbrella/configurations/frame_z{i+36}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/umbrella/plumed/plumed_{i+36}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+36},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

z=-32 | rep=1 frame=98 matches=1
z=-31 | rep=1 frame=136 matches=5
z=-30 | rep=1 frame=12 matches=12
z=-29 | rep=1 frame=93 matches=7
z=-28 | rep=1 frame=138 matches=4
z=-27 | rep=1 frame=112 matches=8
z=-26 | rep=1 frame=169 matches=3
z=-25 | rep=1 frame=63 matches=2
z=-24 | rep=1 frame=175 matches=2
z=-23 | rep=1 frame=4470 matches=3
z=-22 | rep=1 frame=4286 matches=7
z=-21 | rep=1 frame=1630 matches=12
z=-20 | rep=1 frame=4852 matches=37
z=-19 | rep=1 frame=3043 matches=69
z=-18 | rep=1 frame=1176 matches=113
z=-17 | rep=1 frame=670 matches=165


In [20]:
target_z_values = np.arange(19, 35, 1)
tolerance = 0.125

mapping_file = "umbrella_snapshot_mapping_bdqunbiased.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/umbrella/configurations/frame_z{i+52}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/umbrella/plumed/plumed_{i+52}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+52},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

z=19 | rep=2 frame=1550 matches=147
z=20 | rep=2 frame=1487 matches=96
z=21 | rep=2 frame=4025 matches=52
z=22 | rep=2 frame=4624 matches=20
z=23 | rep=0 frame=531 matches=21
z=24 | rep=0 frame=428 matches=23
z=25 | rep=0 frame=195 matches=31
z=26 | rep=0 frame=89 matches=33
z=27 | rep=2 frame=355 matches=36
z=28 | rep=2 frame=477 matches=30
z=29 | rep=2 frame=138 matches=28
z=30 | rep=2 frame=255 matches=29
z=31 | rep=2 frame=261 matches=15
z=32 | rep=2 frame=454 matches=12
z=33 | rep=2 frame=455 matches=5
z=34 | rep=2 frame=505 matches=5


In [ ]:
BDN_com_all_reps = np.empty((len(files), 5001, 3))

for i, simulation in enumerate(files):
    topology, trajectory = simulation
    u = mda.Universe(topology, trajectory)
    print(u.trajectory)
    BDN_com_all_reps[i, :, :] = get_com(u, sel = "resname BDN", ref = "resname POPE")